In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from huggingface_hub import HfFileSystem
import polars as pl

fs = HfFileSystem()


with fs.open("datasets/OpenDataDetector/ColliderML-Release-1/data/ttbar_pu0_particles/train-00000-of-00100.parquet", "rb") as f:
    particles = pl.read_parquet(f)

with fs.open("datasets/OpenDataDetector/ColliderML-Release-1/data/ttbar_pu0_calo_hits/train-00000-of-00100.parquet", "rb") as f:
    calo_hits = pl.read_parquet(f)

with fs.open("datasets/OpenDataDetector/ColliderML-Release-1/data/ttbar_pu0_tracks/train-00000-of-00100.parquet", "rb") as f:
    tracks = pl.read_parquet(f)


In [3]:
particles_old = particles.clone()

In [18]:
particles = particles_old

In [4]:
def add_orphan_mask(df: pl.DataFrame) -> pl.DataFrame:
    print("Computing parent existence mask...")
    # if is_parent_missing exist in df, drop it
    if "is_parent_missing" in df.columns:
        df = df.drop("is_parent_missing")
    
    # 1. Build Lookup Table
    valid_ids_lookup = (
        df.lazy()
        .select(["event_id", "particle_id"])
        .explode("particle_id")
        .rename({"particle_id": "valid_pid"})
        # Ensure ID types match (Int64 vs Int64)
        .with_columns(pl.col("valid_pid").cast(pl.Int64))
        .unique()
        # --- THE FIX: Add a tracer column ---
        # We need this because 'valid_pid' gets dropped during the join.
        .with_columns(pl.lit(True).alias("found_in_event")) 
    )

    # 2. Flatten Parent IDs
    # (Assuming you already ran cast_parent_id_to_int64, so parent_id is Int64)
    parents_flat = (
        df.lazy()
        .select(["event_id", "parent_id"])
        .explode("parent_id")
        .with_row_index("original_order")
    )

    # 3. Join
    matched = parents_flat.join(
        valid_ids_lookup,
        left_on=["event_id", "parent_id"],
        right_on=["event_id", "valid_pid"],
        how="left"
    )

    # 4. Check the Tracer
    result_mask = (
        matched
        .sort("original_order")
        .with_columns(
            # If 'found_in_event' is Null, the join failed -> Parent Missing
            pl.col("found_in_event").is_null().alias("is_parent_missing")
        )
        .group_by("event_id", maintain_order=True)
        .agg(pl.col("is_parent_missing"))
    )

    # 5. Merge back
    return (
        df.lazy()
        .join(result_mask, on="event_id", how="left")
        .collect(streaming=True)
    )

In [5]:
def add_created_inside_calo_mask(particles: pl.DataFrame) -> pl.DataFrame:
    r_xy_sq_threshold = 1250 ** 2
    z_threshold = 3200
    r_min_barrel = 315**2
    

    # 2. Create the Mask Calculation Query
    # We use a separate LazyFrame to calculate masks. This ensures we don't 
    # explode the massive columns (particle_id, parents, etc.) in RAM.
    mask_query = (
        particles.lazy()
        .select(["event_id", "vx", "vy", "vz"]) # Project only what is needed
        .explode(["vx", "vy", "vz"])            # Flatten
        .select([
            pl.col("event_id"),
            (
                # Logic: (vx^2 + vy^2) > 1250^2  OR  
                ((pl.col("vx").pow(2) + pl.col("vy").pow(2)) > r_xy_sq_threshold)
                | 
                ((pl.col("vz").abs() > z_threshold) & ((pl.col("vx").pow(2) + pl.col("vy").pow(2)) > r_min_barrel))
            ).alias("created_inside_calo")
        ])
        # IMPORTANT: maintain_order=True guarantees the mask list 
        # aligns perfectly index-by-index with particle_id list
        .group_by("event_id", maintain_order=True)
        .agg(pl.col("created_inside_calo"))
    )

    # 3. Join back to original data and Collect
    # We use join(how="left") to attach the new column.
    result = (
        particles.lazy()
        .join(mask_query, on="event_id", how="left")
        # Handle cases where an event might have empty lists (join results in null)
        .with_columns(pl.col("created_inside_calo").fill_null([]))
        .collect(streaming=True)
    )

    return result

In [6]:
def add_eta_and_phi(particles: pl.DataFrame) -> pl.DataFrame:
    """
    Adds 'eta' and 'phi' columns calculated from momentum components (px, py, pz).
    
    Formulas:
    pt = sqrt(px^2 + py^2)
    phi = arctan2(py, px)
    theta = arctan2(pt, pz)
    eta = -ln(tan(theta / 2))
    """
    # Calculate eta and phi on flattened data to be memory efficient
    calculations = (
        particles.lazy()
        .select(["event_id", "px", "py", "pz"])
        .explode(["px", "py", "pz"])
        .with_columns(
            (pl.col("px").pow(2) + pl.col("py").pow(2)).sqrt().alias("pt"),
            pl.arctan2(pl.col("py"), pl.col("px")).alias("phi")
        )
        .with_columns(
            pl.arctan2(pl.col("pt"), pl.col("pz")).alias("theta")
        )
        .with_columns(
            (-((pl.col("theta") / 2).tan().log())).alias("eta")
        )
        .group_by("event_id", maintain_order=True)
        .agg([
            pl.col("eta"),
            pl.col("phi"),
            pl.col("pt")
        ])
    )

    return (
        particles.lazy()
        .join(calculations, on="event_id", how="left")
        .with_columns([
            pl.col("eta").fill_null([]),
            pl.col("phi").fill_null([]),
            pl.col("pt").fill_null([])
        ])
        .collect(streaming=True)
    )

In [20]:
def add_target_mask(particles: pl.DataFrame) -> pl.DataFrame:
    target_particles = (
    particles.lazy()
    .select('event_id', 'particle_id', 'is_parent_missing', 'created_inside_calo')
    .explode(['particle_id',  'is_parent_missing', 'created_inside_calo'])
    .join(
        particles.lazy()
        .select('event_id',  'parent_id', 'is_parent_missing', 'created_inside_calo')
        .explode([ 'parent_id', 'is_parent_missing', 'created_inside_calo']),
        left_on=['event_id', 'particle_id'],
        right_on=['event_id', 'parent_id'],
        how='inner',
        suffix="_child"
    )
    .filter(((~pl.col('created_inside_calo')) &
             (pl.col('created_inside_calo_child'))))
.select(['event_id', 'particle_id'])
    .unique()
    .with_columns(pl.lit(True).alias('is_target_particle')))
    # 2. Join back to original data efficiently
    return (
        particles.lazy()
        .select(["event_id", "particle_id"])
        .explode("particle_id")
        .join(
            target_particles,
            on=["event_id", "particle_id"],
            how="left"
        )
        .with_columns(pl.col("is_target_particle").fill_null(False))
        .group_by("event_id", maintain_order=True)
        .agg(pl.col("is_target_particle"))
        .join(
            particles.lazy(),
            on="event_id",
            how="inner"
        )
        .collect(streaming=True)
    )


In [21]:
particles = add_orphan_mask(particles)
particles = add_created_inside_calo_mask(particles)
particles = add_eta_and_phi(particles)
particles = add_target_mask(particles)

Computing parent existence mask...


/tmp/ipykernel_2213063/87548069.py:54: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)
/tmp/ipykernel_2213063/4128779591.py:36: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)
/tmp/ipykernel_2213063/729158228.py:42: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)
/tmp/ipykernel_2213063/1618661146.py:38: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


In [ ]:
def histogram_ht_ratio(particles: pl.DataFrame, eta_cut:float, pt_cut:float)-> pl.DataFrame:
    # sum pt of target particles within eta cut, pt cut
    

In [ ]:
(
    particles.lazy()
    .select('event_id', 'particle_id', 'is_parent_missing', 'created_inside_calo')
    .explode(['particle_id',  'is_parent_missing', 'created_inside_calo'])
    .join(
        particles.lazy()
        .select('event_id',  'parent_id', 'is_parent_missing', 'created_inside_calo')
        .explode([ 'parent_id', 'is_parent_missing', 'created_inside_calo']),
        left_on=['event_id', 'particle_id'],
        right_on=['event_id', 'parent_id'],
        how='inner',
        suffix="_child"
    )
    .filter(
        (
            (~pl.col('created_inside_calo')) &
            (pl.col('created_inside_calo_child'))

         )
    )).collect(streaming=True)

/tmp/ipykernel_2213063/187557707.py:20: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  )).collect(streaming=True)


event_id,particle_id,is_parent_missing,created_inside_calo,is_parent_missing_child,created_inside_calo_child
u32,u64,bool,bool,bool,bool
629,497,false,false,false,true
629,1905,false,false,false,true
629,1905,false,false,false,true
629,1905,false,false,false,true
629,2105,false,false,false,true
…,…,…,…,…,…
197,2804,false,false,false,true
197,2804,false,false,false,true
197,2804,false,false,false,true
